In [1]:
# Add realistic substructure

# chromatin texture in nucleus

# cristae-like internal mito texture

# ER/Golgi tubular networks

# ribosome-like granularity

# sparse dense granules / nanoparticles

# Add acquisition realism

# Poisson counting noise

# detector blur / PSF

# angular jitter

# missing wedge

# ring artifacts

# limited dynamic range

# If you want a more physical version

# A good next step is to split each compartment into:

# density

# elemental composition

# energy-dependent attenuation

In [19]:
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple

from scipy.ndimage import gaussian_filter, rotate, distance_transform_edt

try:
    import xraylib
except ImportError:
    xraylib = None


In [20]:
# -----------------------------
# Labels
# -----------------------------
LABEL_BG = 0
LABEL_WATER = 0   # background is water
LABEL_CYTOPLASM = 1
LABEL_MEMBRANE = 2
LABEL_NUCLEUS = 3
LABEL_NUCLEOLUS = 4
LABEL_MITO = 5
LABEL_VESICLE = 6
LABEL_LIPID = 7


@dataclass
class Material:
    name: str
    density_g_cm3: float
    composition: Dict[str, float]   # mass fractions


# Hydrated compositions
COMPARTMENT_COMPOSITION = {
    "cytoplasm": {"H": 0.10, "C": 0.14, "N": 0.03, "O": 0.72, "P": 0.005, "S": 0.005},
    "membrane":  {"H": 0.10, "C": 0.68, "N": 0.02, "O": 0.18, "P": 0.015, "S": 0.005},
    "nucleus":   {"H": 0.10, "C": 0.18, "N": 0.05, "O": 0.63, "P": 0.03,  "S": 0.01},
    "protein": {"H": 0.065, "C": 0.53, "N": 0.16, "O": 0.23, "P": 0.00, "S": 0.00},
    "nucleicacid": {"H": 0.042, "C": 0.302, "N": 0.149, "O": 0.424, "P": 0.082, "S": 0.00},
}

# Reuse / slight extensions where needed
NUCLEOLUS_COMPOSITION = {"H": 0.10, "C": 0.20, "N": 0.06, "O": 0.58, "P": 0.045, "S": 0.015}
MITO_COMPOSITION      = {"H": 0.10, "C": 0.24, "N": 0.05, "O": 0.57, "P": 0.025, "S": 0.015}
VESICLE_COMPOSITION   = {"H": 0.1119, "O": 0.8881}  # water-like
LIPID_COMPOSITION     = {"H": 0.121, "C": 0.766, "N": 0.00, "O": 0.113, "P": 0.00}
WATER_COMPOSITION     = {"H": 0.1119, "O": 0.8881}

MATERIALS: Dict[int, Material] = {
    LABEL_WATER:     Material("water",         1.00, WATER_COMPOSITION),
    LABEL_CYTOPLASM: Material("cytoplasm",     1.03, COMPARTMENT_COMPOSITION["cytoplasm"]),
    LABEL_MEMBRANE:  Material("membrane",      1.10, COMPARTMENT_COMPOSITION["membrane"]),
    LABEL_NUCLEUS:   Material("nucleus",       1.06, COMPARTMENT_COMPOSITION["nucleus"]),
    LABEL_NUCLEOLUS: Material("nucleolus",     1.10, NUCLEOLUS_COMPOSITION),
    LABEL_MITO:      Material("mitochondrion", 1.08, MITO_COMPOSITION),
    LABEL_VESICLE:   Material("vesicle",       1.00, VESICLE_COMPOSITION),
    LABEL_LIPID:     Material("lipid_droplet", 0.92, LIPID_COMPOSITION),
}

# --- Additional organelle labels ---
LABEL_NENV = 8   # nuclear envelope (double membrane)
LABEL_ER   = 9   # endoplasmic reticulum fragments

NENV_COMPOSITION = {"H": 0.10, "C": 0.60, "N": 0.03, "O": 0.22, "P": 0.025, "S": 0.025}
ER_COMPOSITION   = {"H": 0.10, "C": 0.20, "N": 0.04, "O": 0.62, "P": 0.025, "S": 0.015}

MATERIALS[LABEL_NENV] = Material("nuclear_envelope",      1.12, NENV_COMPOSITION)
MATERIALS[LABEL_ER]   = Material("endoplasmic_reticulum", 1.05, ER_COMPOSITION)


In [21]:
ELEMENT_Z = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "P": 15,
    "S": 16,
}


def mass_attenuation_coefficient_from_composition(composition: Dict[str, float], energy_eV: float) -> float:
    """
    Return mass attenuation coefficient (mu/rho) in cm^2/g
    from elemental mass fractions using xraylib.
    """
    if xraylib is None:
        raise ImportError(
            "xraylib is required for composition-based mu calculation. "
            "Install with: pip install xraylib"
        )

    energy_keV = energy_eV / 1000.0
    mu_over_rho_cm2_g = 0.0

    total_w = sum(composition.values())
    if not np.isclose(total_w, 1.0, atol=1e-6):
        composition = {k: v / total_w for k, v in composition.items()}

    for elem, weight_fraction in composition.items():
        Z = ELEMENT_Z[elem]
        mu_over_rho_cm2_g += weight_fraction * xraylib.CS_Total(Z, energy_keV)

    return mu_over_rho_cm2_g


def linear_attenuation_um_inv(density_g_cm3: float, composition: Dict[str, float], energy_eV: float) -> float:
    """
    Convert density * mass attenuation into linear attenuation in 1/um.

    mu [cm^-1] = rho [g/cm^3] * (mu/rho) [cm^2/g]
    mu [um^-1] = mu [cm^-1] / 1e4
    """
    mu_over_rho = mass_attenuation_coefficient_from_composition(composition, energy_eV)
    mu_cm_inv = density_g_cm3 * mu_over_rho
    mu_um_inv = mu_cm_inv / 1.0e4
    return mu_um_inv


In [22]:
ENERGY_EV = 520.0

MU_520_TABLE = {
    label: linear_attenuation_um_inv(mat.density_g_cm3, mat.composition, ENERGY_EV)
    for label, mat in MATERIALS.items()
}


In [23]:
# -----------------------------
# Geometry helpers
# -----------------------------
def ellipsoid_mask(
    shape: Tuple[int, int, int],
    center: Tuple[float, float, float],
    radii: Tuple[float, float, float],
    rotation_deg_xyz: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> np.ndarray:
    """
    Create a rotated ellipsoid mask in a (z, y, x) volume.
    Rotation order: x, y, z.
    """
    z, y, x = np.indices(shape)
    zz = z - center[0]
    yy = y - center[1]
    xx = x - center[2]

    pts = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=0)

    rx, ry, rz = np.deg2rad(rotation_deg_xyz)

    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx),  np.cos(rx)],
    ])
    Ry = np.array([
        [ np.cos(ry), 0, np.sin(ry)],
        [0, 1, 0],
        [-np.sin(ry), 0, np.cos(ry)],
    ])
    Rz = np.array([
        [np.cos(rz), -np.sin(rz), 0],
        [np.sin(rz),  np.cos(rz), 0],
        [0, 0, 1],
    ])

    R = Rz @ Ry @ Rx
    pts_rot = R.T @ pts

    x2 = pts_rot[0].reshape(shape)
    y2 = pts_rot[1].reshape(shape)
    z2 = pts_rot[2].reshape(shape)

    a, b, c = radii
    return (x2 / a) ** 2 + (y2 / b) ** 2 + (z2 / c) ** 2 <= 1.0


def random_point_in_ellipsoid(
    center: Tuple[float, float, float],
    radii: Tuple[float, float, float],
    rng: np.random.Generator,
    shrink: float = 1.0,
) -> Tuple[float, float, float]:
    """Sample a random point inside an ellipsoid."""
    while True:
        p = rng.uniform(-1.0, 1.0, size=3)
        if np.sum(p**2) <= 1.0:
            break

    scaled = p * np.array(radii) * shrink
    return (
        center[0] + scaled[2],
        center[1] + scaled[1],
        center[2] + scaled[0],
    )


def add_region(label_volume: np.ndarray, mask: np.ndarray, label: int, overwrite: bool = True) -> None:
    """Write a label into the label volume."""
    if overwrite:
        label_volume[mask] = label
    else:
        label_volume[(mask) & (label_volume == LABEL_BG)] = label


In [29]:
def build_cell_phantom(
    voxel_um: float = 0.015,
    shape: Tuple[int, int, int] = None,
    full_resolution: bool = False,
    n_mito: int = 18,
    n_vesicles: int = 35,
    n_lipid: int = 5,
    n_er: int = 14,
    seed: int = 42,
) -> Dict[str, np.ndarray]:
    """
    Build a synthetic 3D cell phantom for soft X-ray absorption imaging at 520 eV.

    Organelles modelled
    -------------------
    * Cytoplasm with ribosome-like granularity
    * Plasma membrane
    * Nucleus with heterochromatin texture
    * Nuclear envelope (double-membrane)
    * Nucleolus
    * Mitochondria with cristae-like sinusoidal density modulation
    * Vesicles (water-like)
    * Lipid droplets (high contrast)
    * ER fragments (rough ER near nucleus)

    Parameters
    ----------
    shape : tuple
        Volume shape (z, y, x).  If None, auto-chosen from full_resolution.
    voxel_um : float
        Voxel edge length in micrometres.
    n_mito : int
        Number of mitochondria.
    n_vesicles : int
        Number of watery vesicles.
    n_lipid : int
        Number of lipid droplets.
    n_er : int
        Number of ER tubule fragments.
    seed : int
        RNG seed.
    """
    rng = np.random.default_rng(seed)

    # ---------- choose shape ----------
    if shape is None:
        if full_resolution:
            z_vox = int(round(10.0 / voxel_um))
            y_vox = int(round(20.0 / voxel_um))
            x_vox = int(round(20.0 / voxel_um))
            shape = (z_vox, y_vox, x_vox)
            print(f"[INFO] full_resolution=True -> shape={shape}  "
                  f"(~{shape[2]*voxel_um:.1f} µm x {shape[1]*voxel_um:.1f} µm x "
                  f"{shape[0]*voxel_um:.1f} µm)")
            print("WARNING: this array may be extremely large (memory/disk). "
                  "Proceed only if you know your resources.")
        else:
            shape = (int(round(3.0 / voxel_um)),
                     int(round(6.0 / voxel_um)),
                     int(round(6.0 / voxel_um)))
            print(f"[INFO] full_resolution=False -> preview shape={shape}  "
                  f"(~{shape[2]*voxel_um:.2f} µm x {shape[1]*voxel_um:.2f} µm x "
                  f"{shape[0]*voxel_um:.2f} µm)")

    z_size_um = shape[0] * voxel_um
    y_size_um = shape[1] * voxel_um
    x_size_um = shape[2] * voxel_um

    # ── initialise with water background ────────────────────────────────────
    label_volume = np.full(shape, LABEL_WATER, dtype=np.uint8)
    cell_center  = ((shape[0] - 1) / 2, (shape[1] - 1) / 2, (shape[2] - 1) / 2)

    if full_resolution:
        cell_radii_um = (3.0, 9.0, 10.0)
    else:
        cell_radii_um = (min(3.0, z_size_um * 0.45),
                         min(9.0, y_size_um * 0.45),
                         min(10.0, x_size_um * 0.45))

    cell_radii = tuple(r / voxel_um for r in cell_radii_um)
    cell_mask  = ellipsoid_mask(shape, cell_center, cell_radii, rotation_deg_xyz=(5, -3, 8))
    add_region(label_volume, cell_mask, LABEL_CYTOPLASM)

    # ── plasma membrane (thin shell) ─────────────────────────────────────────
    membrane_thickness_vox = max(1.0, 0.10 / voxel_um)
    inside_dist    = distance_transform_edt(cell_mask)
    membrane_mask  = cell_mask & (inside_dist <= membrane_thickness_vox)
    cytoplasm_mask = cell_mask & (~membrane_mask)
    add_region(label_volume, cytoplasm_mask, LABEL_CYTOPLASM)
    add_region(label_volume, membrane_mask,  LABEL_MEMBRANE)

    # ── nucleus ──────────────────────────────────────────────────────────────
    nucleus_center = (
        cell_center[0] - (0.3 / voxel_um),
        cell_center[1] + (0.8 / voxel_um),
        cell_center[2] - (1.0 / voxel_um),
    )
    nucleus_radii = (1.8 / voxel_um, 3.8 / voxel_um, 4.2 / voxel_um)
    nucleus_mask  = ellipsoid_mask(shape, nucleus_center, nucleus_radii,
                                   rotation_deg_xyz=(0, 10, -18))
    nucleus_mask &= cytoplasm_mask
    add_region(label_volume, nucleus_mask, LABEL_NUCLEUS)

    # ── nuclear envelope (thin double membrane ~40 nm) ────────────────────────
    nenv_thick_vox = max(1.0, 0.04 / voxel_um)
    dist_in_nuc    = distance_transform_edt(nucleus_mask)
    nenv_mask      = nucleus_mask & (dist_in_nuc <= nenv_thick_vox)
    add_region(label_volume, nenv_mask, LABEL_NENV)

    # ── nucleolus ────────────────────────────────────────────────────────────
    nucleolus_center = (
        nucleus_center[0] + (0.2 / voxel_um),
        nucleus_center[1] - (0.5 / voxel_um),
        nucleus_center[2] + (0.4 / voxel_um),
    )
    nucleolus_radii = (0.7 / voxel_um, 1.0 / voxel_um, 1.0 / voxel_um)
    nucleolus_mask  = ellipsoid_mask(shape, nucleolus_center, nucleolus_radii,
                                     rotation_deg_xyz=(0, 0, 22))
    nucleolus_mask &= nucleus_mask
    add_region(label_volume, nucleolus_mask, LABEL_NUCLEOLUS)

    # ── mitochondria (random elongated ellipsoids) ───────────────────────────
    for _ in range(n_mito):
        for _attempt in range(120):
            c  = random_point_in_ellipsoid(cell_center, cell_radii, rng, shrink=0.78)
            dz = (c[0] - nucleus_center[0]) / nucleus_radii[0]
            dy = (c[1] - nucleus_center[1]) / nucleus_radii[1]
            dx = (c[2] - nucleus_center[2]) / nucleus_radii[2]
            if dz**2 + dy**2 + dx**2 < 1.2:
                continue
            mito_radii = (
                rng.uniform(max(2.0, 0.15 / voxel_um), max(4.0, 0.35 / voxel_um)),
                rng.uniform(max(2.0, 0.20 / voxel_um), max(5.0, 0.45 / voxel_um)),
                rng.uniform(max(4.0, 0.60 / voxel_um), max(10.0, 1.50 / voxel_um)),
            )
            rot        = (rng.uniform(-60, 60), rng.uniform(-60, 60), rng.uniform(-180, 180))
            mito_mask  = ellipsoid_mask(shape, c, mito_radii, rot)
            mito_mask &= cytoplasm_mask
            mito_mask &= ~nucleus_mask
            if np.count_nonzero(mito_mask) < 20:
                continue
            add_region(label_volume, mito_mask, LABEL_MITO)
            break

    # ── vesicles ─────────────────────────────────────────────────────────────
    for _ in range(n_vesicles):
        for _attempt in range(80):
            c      = random_point_in_ellipsoid(cell_center, cell_radii, rng, shrink=0.85)
            ves_r  = rng.uniform(0.03, 0.12) / voxel_um
            ves_m  = ellipsoid_mask(shape, c, (ves_r, ves_r, ves_r))
            ves_m &= cytoplasm_mask
            ves_m &= ~nucleus_mask
            if np.count_nonzero(ves_m) < 8:
                continue
            add_region(label_volume, ves_m, LABEL_VESICLE)
            break

    # ── lipid droplets ───────────────────────────────────────────────────────
    for _ in range(n_lipid):
        for _attempt in range(80):
            c     = random_point_in_ellipsoid(cell_center, cell_radii, rng, shrink=0.72)
            ld_r  = rng.uniform(0.08, 0.22) / voxel_um
            ld_m  = ellipsoid_mask(shape, c, (ld_r, ld_r, ld_r))
            ld_m &= cytoplasm_mask
            ld_m &= ~nucleus_mask
            if np.count_nonzero(ld_m) < 15:
                continue
            add_region(label_volume, ld_m, LABEL_LIPID)
            break

    # ── ER tubule fragments (rough ER near nucleus) ──────────────────────────
    for _ in range(n_er):
        for _attempt in range(80):
            ang   = rng.uniform(0.0, 2 * np.pi)
            elev  = rng.uniform(-np.pi / 3, np.pi / 3)
            scale = rng.uniform(1.05, 1.6)
            c = (
                float(np.clip(nucleus_center[0] + np.sin(elev) * nucleus_radii[0] * scale,
                               1, shape[0] - 2)),
                float(np.clip(nucleus_center[1] + np.cos(elev) * np.sin(ang) * nucleus_radii[1] * scale,
                               1, shape[1] - 2)),
                float(np.clip(nucleus_center[2] + np.cos(elev) * np.cos(ang) * nucleus_radii[2] * scale,
                               1, shape[2] - 2)),
            )
            er_radii = (
                rng.uniform(max(1.0, 0.03 / voxel_um), max(2.0, 0.07 / voxel_um)),
                rng.uniform(max(1.0, 0.05 / voxel_um), max(2.5, 0.12 / voxel_um)),
                rng.uniform(max(3.0, 0.15 / voxel_um), max(7.0, 0.40 / voxel_um)),
            )
            rot   = (rng.uniform(-90, 90), rng.uniform(-90, 90), rng.uniform(-180, 180))
            er_m  = ellipsoid_mask(shape, c, er_radii, rot)
            er_m &= cytoplasm_mask
            er_m &= ~nucleus_mask
            if np.count_nonzero(er_m) < 6:
                continue
            add_region(label_volume, er_m, LABEL_ER)
            break

    # ── base density and mu maps ─────────────────────────────────────────────
    if xraylib is None:
        raise ImportError("xraylib is required. Install with: pip install xraylib")

    density_volume = np.zeros(shape, dtype=np.float32)
    mu_volume      = np.zeros(shape, dtype=np.float32)

    MU_TABLE = {
        lbl: linear_attenuation_um_inv(mat.density_g_cm3, mat.composition, ENERGY_EV)
        for lbl, mat in MATERIALS.items()
    }

    for lbl, mat in MATERIALS.items():
        mask = label_volume == lbl
        density_volume[mask] = mat.density_g_cm3
        mu_volume[mask]      = MU_TABLE[lbl]

    # ── intra-compartment heterogeneity (base noise) ─────────────────────────
    base_noise = rng.normal(0.0, 1.0, size=shape).astype(np.float32)
    base_noise = gaussian_filter(base_noise, sigma=3.0)
    base_noise /= base_noise.std() + 1e-8

    _perturbations = {
        LABEL_CYTOPLASM: 0.015,
        LABEL_MEMBRANE:  0.006,
        LABEL_NUCLEUS:   0.012,
        LABEL_NUCLEOLUS: 0.008,
        LABEL_NENV:      0.005,
        LABEL_MITO:      0.010,
        LABEL_VESICLE:   0.003,
        LABEL_LIPID:     0.004,
        LABEL_ER:        0.007,
        LABEL_WATER:     0.001,
    }
    for lbl, amp in _perturbations.items():
        m = label_volume == lbl
        density_volume[m] += amp * base_noise[m]

    density_volume = np.clip(density_volume, 0.80, 1.45)

    # recompute mu from perturbed density (composition fixed)
    for lbl, mat in MATERIALS.items():
        m = label_volume == lbl
        muo = mass_attenuation_coefficient_from_composition(mat.composition, ENERGY_EV)
        mu_volume[m] = density_volume[m] * muo / 1.0e4

    # small direct mu composition noise
    for lbl, amp in [(LABEL_CYTOPLASM, 0.015), (LABEL_NUCLEUS, 0.020), (LABEL_MITO, 0.020)]:
        m = label_volume == lbl
        mu_volume[m] += amp * base_noise[m]

    # ── cristae-like texture inside mitochondria ─────────────────────────────
    mito_m = label_volume == LABEL_MITO
    if mito_m.any():
        period_vox  = max(3.0, 0.35 / voxel_um)
        z_arr       = np.arange(shape[0], dtype=np.float32)[:, None, None]
        cristae_mod = np.broadcast_to(0.018 * np.sin(2 * np.pi * z_arr / period_vox), shape)
        density_volume[mito_m] += cristae_mod[mito_m]
        muo_m = mass_attenuation_coefficient_from_composition(
            MATERIALS[LABEL_MITO].composition, ENERGY_EV)
        mu_volume[mito_m] = density_volume[mito_m] * muo_m / 1.0e4

    # ── heterochromatin texture (denser near nuclear periphery) ──────────────
    nuc_m = label_volume == LABEL_NUCLEUS
    if nuc_m.any():
        chrom_noise = rng.normal(0.0, 1.0, size=shape).astype(np.float32)
        chrom_noise = gaussian_filter(chrom_noise, sigma=max(2.0, 0.25 / voxel_um))
        chrom_noise /= chrom_noise.std() + 1e-8
        dist_nuc    = distance_transform_edt(nuc_m)
        # higher weight near nuclear envelope -> heterochromatin ring
        bw          = np.exp(-dist_nuc / max(3.0, 0.5 / voxel_um)).astype(np.float32)
        density_volume[nuc_m] += (0.025 * bw * chrom_noise)[nuc_m]
        muo_n = mass_attenuation_coefficient_from_composition(
            MATERIALS[LABEL_NUCLEUS].composition, ENERGY_EV)
        mu_volume[nuc_m] = density_volume[nuc_m] * muo_n / 1.0e4

    # ── ribosome-like granularity in cytoplasm ────────────────────────────────
    cyto_m = label_volume == LABEL_CYTOPLASM
    if cyto_m.any():
        ribo_noise = rng.standard_normal(size=shape).astype(np.float32)
        ribo_noise = gaussian_filter(ribo_noise, sigma=0.7)
        ribo_noise /= ribo_noise.std() + 1e-8
        density_volume[cyto_m] += (0.006 * ribo_noise)[cyto_m]
        muo_c = mass_attenuation_coefficient_from_composition(
            MATERIALS[LABEL_CYTOPLASM].composition, ENERGY_EV)
        mu_volume[cyto_m] = density_volume[cyto_m] * muo_c / 1.0e4

    density_volume = np.clip(density_volume, 0.80, 1.45)
    mu_volume      = np.clip(mu_volume, 0.0, None)

    return {
        "label_volume":   label_volume,
        "density_volume": density_volume,
        "mu_volume":      mu_volume,
        "voxel_um":       float(voxel_um),
        "shape":          shape,
        "fov_um":         (z_size_um, y_size_um, x_size_um),
    }


In [ ]:
def plot_results_slices(
    phantom: dict,
    save_path: str = "cell_results_slices.png",
) -> None:
    """
    Plot the 3D cell phantom as multi-orientation slice panels and save to file.

    Layout (4 rows × 3 columns)
    ----------------------------
    Row 0 – Segmentation label map (XY, XZ, YZ centre slices)
    Row 1 – Mass density [g/cm³]   (XY, XZ, YZ centre slices)
    Row 2 – Linear attenuation µ [1/µm] (XY, XZ, YZ centre slices)
    Row 3 – Beer-Lambert projections along Z, Y, X axes

    Parameters
    ----------
    phantom : dict
        Output dictionary from build_cell_phantom().
    save_path : str
        File path for the saved figure (PNG).
    """
    import matplotlib.patches as mpatches
    from matplotlib.colors import ListedColormap, BoundaryNorm

    labels   = phantom["label_volume"]
    density  = phantom["density_volume"]
    mu       = phantom["mu_volume"]
    voxel_um = phantom["voxel_um"]

    cz = labels.shape[0] // 2
    cy = labels.shape[1] // 2
    cx = labels.shape[2] // 2

    # Beer-Lambert projections along all three axes
    proj_z = project_absorption(mu, voxel_um=voxel_um, axis=0)
    proj_y = project_absorption(mu, voxel_um=voxel_um, axis=1)
    proj_x = project_absorption(mu, voxel_um=voxel_um, axis=2)

    # ── label colourmap ──────────────────────────────────────────────────────
    _PALETTE = [
        "#111122",  # 0  water / background
        "#4a90d9",  # 1  cytoplasm
        "#e8c97a",  # 2  membrane
        "#7b68ee",  # 3  nucleus
        "#ff6b6b",  # 4  nucleolus
        "#48bb78",  # 5  mitochondria
        "#81e6d9",  # 6  vesicle
        "#f6ad55",  # 7  lipid droplet
        "#ff9ff3",  # 8  nuclear envelope
        "#feca57",  # 9  ER
    ]
    n_lbl    = int(labels.max()) + 1
    palette  = (_PALETTE + ["#aaaaaa"] * 10)[:max(n_lbl, len(_PALETTE))]
    lbl_cmap = ListedColormap(palette)
    lbl_norm = BoundaryNorm(np.arange(-0.5, len(palette) + 0.5), len(palette))

    fig, axes = plt.subplots(4, 3, figsize=(14, 18))
    fig.suptitle("3D Cell Phantom – Results (520 eV)", fontsize=14,
                 fontweight="bold", y=0.998)

    col_titles = ["XY  (z = centre)", "XZ  (y = centre)", "YZ  (x = centre)"]

    # ── Row 0: label maps ────────────────────────────────────────────────────
    lbl_slices = [labels[cz], labels[:, cy], labels[:, :, cx]]
    for j, (sl, ct) in enumerate(zip(lbl_slices, col_titles)):
        ax = axes[0, j]
        ax.imshow(sl, cmap=lbl_cmap, norm=lbl_norm, origin="lower",
                  interpolation="nearest")
        ax.set_title(ct, fontsize=9)
        if j == 0:
            ax.set_ylabel("Segmentation\nlabels", fontsize=9)
        ax.tick_params(labelsize=7)

    present_lbls = sorted(np.unique(labels))
    patches = []
    for lbl in present_lbls:
        mat   = MATERIALS.get(int(lbl))
        name  = mat.name if mat else f"label {lbl}"
        color = palette[lbl] if lbl < len(palette) else "#aaaaaa"
        patches.append(mpatches.Patch(color=color, label=name))
    axes[0, 2].legend(handles=patches, fontsize=7, loc="upper right",
                      framealpha=0.85)

    # ── Row 1: density [g/cm³] ───────────────────────────────────────────────
    den_slices = [density[cz], density[:, cy], density[:, :, cx]]
    occ        = density[labels > 0]
    vdn, vdx   = (float(occ.min()), float(occ.max())) if occ.size else (0.9, 1.2)
    for j, (sl, ct) in enumerate(zip(den_slices, col_titles)):
        ax = axes[1, j]
        im = ax.imshow(sl, cmap="viridis", vmin=vdn, vmax=vdx, origin="lower")
        ax.set_title(ct, fontsize=9)
        if j == 0:
            ax.set_ylabel("Density\n[g/cm³]", fontsize=9)
        ax.tick_params(labelsize=7)
        plt.colorbar(im, ax=ax, shrink=0.75, format="%.3f")

    # ── Row 2: linear attenuation µ [1/µm] ──────────────────────────────────
    mu_slices = [mu[cz], mu[:, cy], mu[:, :, cx]]
    mu_occ    = mu[labels > 0]
    vmn, vmx  = (0.0, float(np.percentile(mu_occ, 99.5))) if mu_occ.size else (0, 0.1)
    for j, (sl, ct) in enumerate(zip(mu_slices, col_titles)):
        ax = axes[2, j]
        im = ax.imshow(sl, cmap="inferno", vmin=vmn, vmax=vmx, origin="lower")
        ax.set_title(ct, fontsize=9)
        if j == 0:
            ax.set_ylabel("Linear attenuation\nµ  [1/µm]", fontsize=9)
        ax.tick_params(labelsize=7)
        plt.colorbar(im, ax=ax, shrink=0.75, format="%.4f")

    # ── Row 3: Beer-Lambert projections ─────────────────────────────────────
    projs       = [proj_z, proj_y, proj_x]
    proj_labels = ["Proj. along Z", "Proj. along Y", "Proj. along X"]
    for j, (proj, plbl) in enumerate(zip(projs, proj_labels)):
        ax = axes[3, j]
        im = ax.imshow(proj, cmap="gray", origin="lower")
        ax.set_title(plbl, fontsize=9)
        if j == 0:
            ax.set_ylabel("Beer-Lambert\nprojection  I/I₀", fontsize=9)
        ax.tick_params(labelsize=7)
        plt.colorbar(im, ax=ax, shrink=0.75, format="%.3f")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


In [30]:
def project_absorption(
    mu_volume: np.ndarray,
    voxel_um: float,
    axis: int = 0,
    I0: float = 1.0,
) -> np.ndarray:
    """
    Beer-Lambert projection for absorption imaging.

    I = I0 * exp(- integral(mu dl))

    mu_volume is in [1/um]
    voxel_um is in [um]
    """
    path_integral = np.sum(mu_volume, axis=axis) * voxel_um
    projection = I0 * np.exp(-path_integral)
    return projection.astype(np.float32)


def rotate_volume_for_angle(volume: np.ndarray, angle_deg: float) -> np.ndarray:
    """Rotate around z-axis for tomography-style acquisition."""
    return rotate(
        volume,
        angle=angle_deg,
        axes=(1, 2),
        reshape=False,
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=True,
    )


def simulate_sinogram(
    mu_volume: np.ndarray,
    voxel_um: float,
    angles_deg: np.ndarray,
) -> np.ndarray:
    """
    Simulate a simple parallel-beam projection stack.
    Output shape: (n_angles, y, x)
    """
    projections = []
    for angle in angles_deg:
        vol_rot = rotate_volume_for_angle(mu_volume, float(angle))
        proj = project_absorption(vol_rot, voxel_um=voxel_um, axis=0)
        projections.append(proj)
    return np.stack(projections, axis=0)

In [31]:
def summarize_volume(label_volume: np.ndarray, density_volume: np.ndarray, mu_volume: np.ndarray) -> None:
    print(f"Volume summary at {ENERGY_EV:.1f} eV")
    print("-" * 76)
    for label, material in MATERIALS.items():
        mask = label_volume == label
        n = np.count_nonzero(mask)
        if n == 0:
            continue
        mean_rho = float(density_volume[mask].mean())
        mean_mu = float(mu_volume[mask].mean())
        print(
            f"{label:2d}  {material.name:14s}  "
            f"voxels={n:10d}  density={mean_rho:.3f} g/cm^3  mu={mean_mu:.4f} 1/um"
        )


In [32]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Checkbox, fixed
from IPython.display import display


def normalize_limits(arr, pmin=1, pmax=99):
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.percentile(vals, [pmin, pmax])
    if vmin == vmax:
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)


def get_slice(vol, axis, index):
    if axis == 0:
        return vol[index, :, :]
    elif axis == 1:
        return vol[:, index, :]
    elif axis == 2:
        return vol[:, :, index]
    else:
        raise ValueError("axis must be 0, 1, or 2")


def show_volume_slice(
    volume,
    labels=None,
    title="volume",
    cmap="gray",
    label_cmap="tab20",
):
    """
    Jupyter-friendly 3D slicer for numpy arrays.

    Parameters
    ----------
    volume : np.ndarray
        3D array to visualize
    labels : np.ndarray or None
        Optional 3D integer label array for overlay
    title : str
        Viewer title
    cmap : str
        Colormap for the main volume
    label_cmap : str
        Colormap for labels overlay
    """
    if volume.ndim != 3:
        raise ValueError(f"volume must be 3D, got shape {volume.shape}")

    if labels is not None:
        if labels.shape != volume.shape:
            raise ValueError("labels must have the same shape as volume")

    default_vmin, default_vmax = normalize_limits(volume)

    axis_widget = Dropdown(
        options=[("z", 0), ("y", 1), ("x", 2)],
        value=0,
        description="Axis:"
    )

    slice_widget = IntSlider(
        min=0,
        max=volume.shape[0] - 1,
        step=1,
        value=volume.shape[0] // 2,
        description="Slice:",
        continuous_update=False
    )

    vmin_widget = FloatSlider(
        min=float(np.nanmin(volume)),
        max=float(np.nanmax(volume)),
        step=(float(np.nanmax(volume)) - float(np.nanmin(volume)) + 1e-9) / 500.0,
        value=default_vmin,
        description="vmin:",
        continuous_update=False,
        readout_format=".4f",
        layout={"width": "90%"}
    )

    vmax_widget = FloatSlider(
        min=float(np.nanmin(volume)),
        max=float(np.nanmax(volume)),
        step=(float(np.nanmax(volume)) - float(np.nanmin(volume)) + 1e-9) / 500.0,
        value=default_vmax,
        description="vmax:",
        continuous_update=False,
        readout_format=".4f",
        layout={"width": "90%"}
    )

    overlay_widget = Checkbox(
        value=(labels is not None),
        description="Overlay labels",
        disabled=(labels is None)
    )

    alpha_widget = FloatSlider(
        min=0.0,
        max=1.0,
        step=0.05,
        value=0.35,
        description="Label α:",
        continuous_update=False
    )

    def update_slice_range(*args):
        ax = axis_widget.value
        slice_widget.max = volume.shape[ax] - 1
        if slice_widget.value > slice_widget.max:
            slice_widget.value = slice_widget.max

    axis_widget.observe(update_slice_range, names="value")

    def viewer(axis, slice_idx, vmin, vmax, overlay, alpha):
        img = get_slice(volume, axis, slice_idx)

        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")

        if labels is not None and overlay:
            lab = get_slice(labels, axis, slice_idx)
            masked_lab = np.ma.masked_where(lab == 0, lab)
            ax.imshow(masked_lab, cmap=label_cmap, alpha=alpha, origin="lower", interpolation="nearest")

        ax.set_title(f"{title} | axis={axis} | slice={slice_idx}")
        ax.set_xlabel("pixel")
        ax.set_ylabel("pixel")
        plt.colorbar(im, ax=ax, shrink=0.8)
        plt.show()

    interact(
        viewer,
        axis=axis_widget,
        slice_idx=slice_widget,
        vmin=vmin_widget,
        vmax=vmax_widget,
        overlay=overlay_widget,
        alpha=alpha_widget,
    )


In [33]:
# -----------------------------
# Example run (default preview)
# -----------------------------
if __name__ == "__main__" or True:
    voxel_um       = 0.05
    full_resolution = False

    phantom = build_cell_phantom(
        voxel_um=voxel_um,
        full_resolution=full_resolution,
        shape=(200, 200, 200),
        seed=42,
    )

    labels   = phantom["label_volume"]
    density  = phantom["density_volume"]
    mu       = phantom["mu_volume"]
    voxel_um = phantom["voxel_um"]
    print(f"phantom shape = {phantom['shape']}, voxel = {voxel_um} µm, "
          f"FOV (um z,y,x) = {phantom['fov_um']}")

    summarize_volume(labels, density, mu)

    # single projection (along z-axis)
    proj0 = project_absorption(mu, voxel_um=voxel_um, axis=0)
    print("projection shape:", proj0.shape,
          "min/max:", float(proj0.min()), float(proj0.max()))

    # small sinogram
    angles = np.linspace(0, 180, 16, endpoint=False)
    sino   = simulate_sinogram(mu, voxel_um=voxel_um, angles_deg=angles)
    print("sinogram shape:", sino.shape)

    # save numpy outputs
    np.save("cell_labels.npy",           labels)
    np.save("cell_density_gcm3.npy",     density)
    np.save("cell_mu_520eV_uminv.npy",   mu)
    np.save("cell_projection_0deg.npy",  proj0)
    np.save("cell_sinogram.npy",         sino)
    print("Saved numpy arrays.")

    # ── static slice visualisation ──────────────────────────────────────────
    plot_results_slices(phantom, save_path="cell_results_slices.png")

    # ── interactive slicer (Jupyter only) ───────────────────────────────────
    try:
        show_volume_slice(mu, labels=labels, title="mu at 520 eV [1/um]", cmap="inferno")
    except Exception as e:
        print("[INFO] interactive slicer unavailable:", e)


phantom shape = (200, 200, 200), voxel = 0.05 µm, FOV (um z,y,x) = (10.0, 10.0, 10.0)
Volume summary at 520.0 eV
----------------------------------------------------------------------------
 0  water           voxels=   5964244  density=1.000 g/cm^3  mu=0.1178 1/um
 2  membrane        voxels=    137030  density=1.100 g/cm^3  mu=0.9997 1/um
 3  nucleus         voxels=    825469  density=1.061 g/cm^3  mu=0.4546 1/um
 4  nucleolus       voxels=     23480  density=1.101 g/cm^3  mu=0.5297 1/um
 5  mitochondrion   voxels=   1048432  density=1.080 g/cm^3  mu=0.5326 1/um
 6  vesicle         voxels=       923  density=1.001 g/cm^3  mu=0.1179 1/um
 7  lipid_droplet   voxels=       422  density=0.918 g/cm^3  mu=0.8738 1/um
projection shape: (200, 200) min/max: 0.00705143203958869 0.30829620361328125
sinogram shape: (16, 200, 200)
Saved: cell_labels.npy, cell_density_gcm3.npy, cell_mu_520eV_uminv.npy, cell_projection_0deg.npy, cell_sinogram.npy


interactive(children=(Dropdown(description='Axis:', options=(('z', 0), ('y', 1), ('x', 2)), value=0), IntSlide…

In [34]:
import tifffile as tiff
import numpy as np
import os

def save_tiff_stack(volume, filename, voxel_um=0.015):
    """
    Save a 3D numpy array as a TIFF stack (Z, Y, X).
    Includes voxel size metadata (in µm).
    """
    volume = np.asarray(volume)

    # Ensure correct dtype
    if volume.dtype == np.float64:
        volume = volume.astype(np.float32)

    # Metadata (ImageJ-compatible)
    metadata = {
        'spacing': voxel_um,   # z spacing
        'unit': 'um'
    }

    tiff.imwrite(
        filename,
        volume,
        imagej=True,
        metadata=metadata
    )
    print(f"Saved: {filename}  shape={volume.shape}  dtype={volume.dtype}")


# Example usage
save_tiff_stack(labels.astype(np.uint8), "cell_labels.tif", voxel_um)
save_tiff_stack(density.astype(np.float32), "cell_density.tif", voxel_um)
save_tiff_stack(mu.astype(np.float32), "cell_mu_520eV.tif", voxel_um)


Saved: cell_labels.tif  shape=(200, 200, 200)  dtype=uint8
Saved: cell_density.tif  shape=(200, 200, 200)  dtype=float32
Saved: cell_mu_520eV.tif  shape=(200, 200, 200)  dtype=float32


In [13]:
show_volume_slice(density, labels=labels, title="Density [g/cm^3]", cmap="viridis")


interactive(children=(Dropdown(description='Axis:', options=(('z', 0), ('y', 1), ('x', 2)), value=0), IntSlide…

In [ ]:
show_volume_slice(mu, labels=labels, title="mu at 520 eV [1/um]", cmap="inferno")
